## Cài đặt thư viện

In [ ]:
!pip install timm einops simple-tokenizer
!pip install termcolor
!pip install -U "pymilvus[milvus-lite]"
!pip install sentence-transformers pandas
!pip install -U bitsandbytes accelerate

ERROR: Could not find a version that satisfies the requirement simple-tokenizer (from versions: none)
ERROR: No matching distribution found for simple-tokenizer


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Xử lý dữ liệu và đẩy vào Milvus (Ingestion)

## Imports & Setup Embedding

In [ ]:
import os
from pymilvus import MilvusClient
from sentence_transformers import SentenceTransformer, CrossEncoder

# Định nghĩa đường dẫn lưu DB trong Google Drive
db_folder = '/content/drive/Shared drives/NLP - Text2Img/Prj_Fin_24_23127082_23127394_23127398_23127527/Build RAG/rag'
db_path = os.path.join(db_folder, "milvus_history.db")
os.makedirs(db_folder, exist_ok=True)

# Khởi tạo Milvus Client trỏ vào file trong Drive
milvus_client = MilvusClient(uri=db_path)

# Load Model Embedding
embed_model = SentenceTransformer('bkai-foundation-models/vietnamese-bi-encoder')

# Load model Reranker
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=512)

print(f"Đã kết nối tới Database tại: {db_path}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Đã kết nối tới Database tại: /content/drive/Shared drives/NLP - Text2Img/Prj_Fin_24_23127082_23127394_23127398_23127527/rag/milvus_history.db


## Load Data & Insert to Milvus

In [ ]:
import pandas as pd
from tqdm import tqdm
import numpy as np
import os

collection_name = "sgk_history"

# KIỂM TRA DỮ LIỆU CŨ
has_collection = milvus_client.has_collection(collection_name)
row_count = 0

if has_collection:
    stats = milvus_client.get_collection_stats(collection_name)
    row_count = stats['row_count']

if has_collection and row_count > 0:
    print(f"Đã tìm thấy dữ liệu cũ trong Drive (Collection: {collection_name}).")
    print(f"Số lượng bản ghi hiện có: {row_count}")
    print("Sẵn sàng sử dụng ngay, bỏ qua bước embedding!")

else:
    # KHỞI TẠO MỚI
    print("Chưa có dữ liệu hoặc collection rỗng. Đang tiến hành khởi tạo mới...")

    # Xóa collection cũ nếu tồn tại
    if has_collection:
        milvus_client.drop_collection(collection_name)

    # Tạo Collection mới
    milvus_client.create_collection(
        collection_name=collection_name,
        dimension=768,
        metric_type="COSINE",
        auto_id=True,
    )

    # ĐỌC FILE DATA
    base_folder = '/content/drive/Shared drives/NLP - Text2Img/Prj_Fin_24_23127082_23127394_23127398_23127527/Build RAG/dataset/'
    file_names = ['ket_qua_67.csv', 'ket_qua_89.csv']

    df_list = []
    for fname in file_names:
        fpath = os.path.join(base_folder, fname)
        try:
            d = pd.read_csv(fpath)
            print(f"Đã đọc file {fname}: {len(d)} dòng.")
            df_list.append(d)
        except FileNotFoundError:
            print(f"LỖI: Không tìm thấy file {fname} tại {base_folder}")

    if df_list:
        full_df = pd.concat(df_list, ignore_index=True)
        print(f"∑ Tổng cộng: {len(full_df)} dòng dữ liệu.")

        data_to_insert = []
        print("Đang tạo vector (Embedding)...")

        for idx, row in tqdm(full_df.iterrows(), total=len(full_df)):
            # Xử lý dữ liệu (tránh lỗi NaN nếu ô trống)
            ocr_text = str(row['ocr_text']) if not pd.isna(row['ocr_text']) else ""
            caption = str(row['caption']) if not pd.isna(row['caption']) else ""
            image_path = str(row['image_path'])
            original_id = str(row['id'])

            # Combine Text để Embedding
            text_to_embed = f"{caption}. {ocr_text}".strip()
            if len(text_to_embed) < 10 or ocr_text == "ERROR_DURING_OCR":
                continue

            # Tạo Vector
            vector = embed_model.encode(text_to_embed).tolist()

            # Lưu vào Milvus (Lưu tách biệt caption và ocr_text để LLM dùng sau này)
            data_to_insert.append({
                "vector": vector,
                "ocr_text": ocr_text,
                "caption": caption,
                "image_path": image_path,
                "original_id": original_id
            })

        # Insert vào Milvus
        if len(data_to_insert) > 0:
            milvus_client.insert(collection_name=collection_name, data=data_to_insert)
            print(f"Đã lưu thành công {len(data_to_insert)} bản ghi vào Milvus!")
        else:
            print("Không có dữ liệu hợp lệ để lưu.")
    else:
        print("Không đọc được file CSV nào cả.")

Đã tìm thấy dữ liệu cũ trong Drive (Collection: sgk_history).
Số lượng bản ghi hiện có: 821
Sẵn sàng sử dụng ngay, bỏ qua bước embedding!


# Load Model Qwen2.5-3B-Instruct

## Load Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "Qwen/Qwen2.5-3B-Instruct"
print(f"Đang tải {model_name} (Mode 4-bit)...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Đã load Qwen 3B thành công!")

Đang tải Qwen/Qwen2.5-3B-Instruct (Mode 4-bit)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Đã load Qwen 3B thành công (VRAM chỉ tốn ~2.5GB)!


# Pipeline RAG & Prompt Engineering
Query -> Milvus -> Context -> Qwen -> Diffusion Prompt

## Define RAG Function

Hàm có RAG

In [ ]:
def generate_diffusion_prompt(user_query, top_k=3, score_threshold=0.2):
    # =========================================================================
    # BƯỚC 1: RETRIEVAL (Truy vấn Milvus - Giữ nguyên)
    # =========================================================================

    query_vector = embed_model.encode(user_query).tolist()

    search_res = milvus_client.search(
        collection_name=collection_name,
        data=[query_vector],
        limit=top_k,
        output_fields=["ocr_text", "caption", "image_path"]
    )
    # =========================================================================
    # BƯỚC 1.5: RE-RANKING (Chấm điểm lại để lọc rác)
    # =========================================================================
    hits = search_res[0]
    rerank_pairs = []

    for hit in hits:
        combined_text = f"{hit['entity']['caption']}. {hit['entity']['ocr_text']}"
        rerank_pairs.append([user_query, combined_text])

    # Model chấm điểm
    if len(rerank_pairs) > 0:
        scores = reranker.predict(rerank_pairs)
        for i, hit in enumerate(hits):
            hit['rerank_score'] = scores[i]
    else:
        scores = []

    # Sắp xếp từ cao xuống thấp
    sorted_hits_all = sorted(hits, key=lambda x: x['rerank_score'], reverse=True)

    # LỌC NGƯỠNG 0.2
    valid_hits = [hit for hit in sorted_hits_all if hit['rerank_score'] >= score_threshold]

    # Cắt lấy Top K
    final_hits = valid_hits[:top_k]

    # =========================================================================
    # BƯỚC 2: PREPARE PROMPT
    # =========================================================================
    context_str = ""
    print(f"\n--- Tìm thấy {len(final_hits)} nguồn dữ liệu ĐẠT CHUẨN (Score >= {score_threshold}): ---")

    if len(final_hits) == 0:
        print(f"CẢNH BÁO: Không có dữ liệu nào đạt điểm liên quan >= {score_threshold}.")
        print("Hệ thống sẽ bỏ qua RAG và dùng kiến thức nội tại (Internal Knowledge) của Qwen.")

    for hit in final_hits:
        entity = hit['entity']
        score = hit['rerank_score']

        print(f"[Score: {score:.4f}] Giữ lại file: {entity['image_path']}")

        context_str += f"--- Source Image: {entity['image_path']} ---\n"
        context_str += f"Caption: {entity['caption']}\n"
        context_str += f"Text Content: {entity['ocr_text']}\n\n"
    print(context_str)

    system_instruction = "You are an AI Image Prompt Expert. You convert Vietnamese historical context into concise ENGLISH prompts for Stable Diffusion. You Output ONLY the prompt."

    user_content = f"""
    RETRIEVED CONTEXT (From Textbooks):
    {context_str}

    USER TOPIC: "{user_query}"

    TASK: Write a text-to-image prompt in ENGLISH for the User Topic.

    CRITICAL INSTRUCTION:
    1. Combine the Retrieved Context with your own INTERNAL KNOWLEDGE of Vietnamese history.
    2. Since the context might be sparse, USE YOUR KNOWLEDGE to fill in missing visual details (architecture, weapons, clothing patterns, atmosphere).
    3. FOCUS HEAVILY ON VISUAL DETAILS: What are they wearing? What are they holding? What is in the background?
    4. The AI model is already fine-tuned for the art style. DO NOT describe the style/lighting. Describe ONLY the SUBJECT and ACTION.
    5. Analyze the USER TOPIC to decribe:
      MODE A: CHARACTER FOCUS (If topic is a specific Person e.g., 'Tran Quoc Tuan', 'Lady Trieu')
      - Main Subject: A single detailed character in the center.
      - Camera: Medium shot or Cowboy shot (from knees up).
      - Details: Describe face, beard, specific armor patterns, helmet, hand gestures holding weapon.
      - Background: Can be a battle, but it must be BLURRED or secondary to the character.

      MODE B: EVENT/BATTLE FOCUS (If topic is an Event e.g., 'Battle of Bach Dang', 'Mongol Invasion')
      - Main Subject: Two armies, crowds, massive scale.
      - Camera: Wide angle view or Aerial view.
      - Details: Focus on flags, many spears, ships, chaos, smoke.
      - Do NOT focus on one individual.

    STRICT CONSTRAINTS:
    1. Language: ENGLISH ONLY.
    2. Format: Comma-separated keywords (Subject Description, Clothing/Accessories, Action, Environment/Background).
    3. Length: Under 100 tokens.
    4. NO PROPER NAMES: Use descriptive terms (e.g., 'a bearded general', 'an army').
    5. VIETNAMESE CONTEXT: Characters must have Asian features. Clothing must be ancient Vietnamese (e.g., 'Giao Linh robe', 'lacquered armor', 'conical hat'). NO European/Chinese fantasy armor.
    6. NO STYLE WORDS: Do NOT use words like 'cinematic lighting', '8k', 'realistic', 'cartoon', 'style'. Just describe the content.

    Example Output (If Mode A - Person):
    a majestic middle-aged Asian general wearing red lacquered armor and fabric helmet, holding a sword, shouting command, heroic pose, medium shot, blurred soldiers in background

    Example Output (If Mode B - Battle):
    chaotic river battlefield, wide angle view, two massive armies clashing, hundreds of Vietnamese soldiers in red armor holding spears, wooden stakes in water, burning ships, smoke
    """

    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_content}
    ]

    # =========================================================================
    # BƯỚC 3: GENERATION (Max 100 tokens)
    # =========================================================================

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.2,
        temperature=None,
        top_p=None
    )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    response = response.replace("\n", ", ").strip()

    return response

Hàm không RAG

In [ ]:
def generate_diffusion_prompt_no_rag(user_query):
    # =========================================================================
    # BƯỚC 1: PREPARE PROMPT (Dựa hoàn toàn vào kiến thức nội tại)
    # =========================================================================

    system_instruction = "You are an AI Image Prompt Expert. You convert Vietnamese historical topics into concise ENGLISH prompts for Stable Diffusion."

    user_content = f"""
    TOPIC (Vietnamese): "{user_query}"

    TASK: Write a text-to-image prompt in ENGLISH for the topic above based on your internal knowledge.

    CRITICAL INSTRUCTION:
    1. USE YOUR KNOWLEDGE to fill in missing visual details (architecture, weapons, clothing patterns, atmosphere).
    2. FOCUS HEAVILY ON VISUAL DETAILS: What are they wearing? What are they holding? What is in the background?
    3. The AI model is already fine-tuned for the art style. DO NOT describe the style/lighting. Describe ONLY the SUBJECT and ACTION.
    4. Analyze the USER TOPIC to decribe:
      MODE A: CHARACTER FOCUS (If topic is a specific Person e.g., 'Tran Quoc Tuan', 'Lady Trieu')
      - Main Subject: A single detailed character in the center.
      - Camera: Medium shot or Cowboy shot (from knees up).
      - Details: Describe face, beard, specific armor patterns, helmet, hand gestures holding weapon.
      - Background: Can be a battle, but it must be BLURRED or secondary to the character.

      MODE B: EVENT/BATTLE FOCUS (If topic is an Event e.g., 'Battle of Bach Dang', 'Mongol Invasion')
      - Main Subject: Two armies, crowds, massive scale.
      - Camera: Wide angle view or Aerial view.
      - Details: Focus on flags, many spears, ships, chaos, smoke.
      - Do NOT focus on one individual.

    STRICT CONSTRAINTS:
    1. Language: ENGLISH ONLY.
    2. Format: Comma-separated keywords (Subject Description, Clothing/Accessories, Action, Environment/Background).
    3. Length: Under 100 tokens.
    4. NO PROPER NAMES: Use descriptive terms (e.g., 'a bearded general', 'an army').
    5. VIETNAMESE CONTEXT: Characters must have Asian features. Clothing must be ancient Vietnamese (e.g., 'Giao Linh robe', 'lacquered armor', 'conical hat'). NO European/Chinese fantasy armor.
    6. NO STYLE WORDS: Do NOT use words like 'cinematic lighting', '8k', 'realistic', 'cartoon', 'style'. Just describe the content.

    Example Output (If Mode A - Person):
    a majestic middle-aged Asian general wearing red lacquered armor and fabric helmet, holding a sword, shouting command, heroic pose, medium shot, blurred soldiers in background

    Example Output (If Mode B - Battle):
    chaotic river battlefield, wide angle view, two massive armies clashing, hundreds of Vietnamese soldiers in red armor holding spears, wooden stakes in water, burning ships, smoke
    """

    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_content}
    ]

    # =========================================================================
    # BƯỚC 2: GENERATION
    # =========================================================================

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.2,
        temperature=None,
        top_p=None
    )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    response = response.replace("\n", ", ").strip()

    return response

# Chạy thử nghiệm

In [ ]:
%%time

user_input = "Trận Bạch Đằng"
print(f"User Input: {user_input}")

result = generate_diffusion_prompt(user_input)
result2 = generate_diffusion_prompt_no_rag(user_input)

User Input: Trận Bạch Đằng


The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Tìm thấy 2 nguồn dữ liệu ĐẠT CHUẨN (Score >= 0.2): ---
[Score: 0.8595] Giữ lại file: /kaggle/input/sgk-6-7/lichsu_diali_67/lich-su-va-dia-li-6_page_100.jpg
[Score: 0.5374] Giữ lại file: /kaggle/input/sgk-6-7/lichsu_diali_67/lich-su-va-dia-ly-7_page_75.jpg
--- Source Image: /kaggle/input/sgk-6-7/lichsu_diali_67/lich-su-va-dia-li-6_page_100.jpg ---
Caption: Hình 1: Tranh minh họa nhân vật Lưu Hoàng Tháo đang chỉ đạo quân đội tiến vào cửa sông Bạch Đằng. Hành động của nhân vật được thể hiện bằng màu đỏ và biểu tượng chỉ đạo.

Hình 2: Sơ đồ lịch sử với các mốc thời gian 905, 907, 931, 938. Các mốc này được đánh dấu bằng màu xanh lá cây và số liệu tương ứng.
Text Content: Cuối năm 938, đoàn thuyền chiến do Lưu Hoàng Tháo chỉ huy tiến vào cửa sông Bạch Đằng. Nhân lúc thuỷ triều lên, Ngô Quyền cho thuyền nhỏ ra khiêu chiến, nhử quân giặc tiến sâu vào cửa sông. Lưu Hoàng Tháo cho quân đuổi theo, vượt qua bãi cọc ngầm.

Dợi khi thuỷ triều rút, Ngô Quyền hạ lệnh tấn công. Quân giặc thua và 

In [ ]:
print("KẾT QUẢ (CÓ RAG):")
print(result)
print("\n" + "="*50)
print("KẾT QUẢ (KHÔNG RAG):")
print("="*50)
print(result2)

KẾT QUẢ (CÓ RAG):
A bloody river battlefield, wide angle view, two massive armies clashing, hundreds of Vietnamese soldiers in red lacquer armor wielding long spears, wooden stakes jutting from water, flaming ships sinking, smoke rising, boats overturned, bodies floating, bloodied rivers, victorious troops advancing, triumphant banners waving

KẾT QUẢ (KHÔNG RAG):
Mode B: Battle, chaotic river battlefield, wide angle view, two massive armies clashing, hundreds of Vietnamese soldiers in red lacquered armor with conical hats holding long spears, flaming arrows raining down, boats sinking in water, smoke rising, broken bridges across river
